# Transformação da camada Bronze para a camada Silver

**Tech Challenge Fase 3** · Pós-Tech em Data Analytics, FIAP
**Base:** State of Data Brazil, edições 2023-2024, 2024-2025 e 2025-2026
**Etapa do pipeline:** harmonização das três edições

Esta é a etapa mais delicada do projeto. As três edições usam padrões de nomenclatura diferentes, e o código da pergunta muda de significado entre elas, o que torna impossível unir as bases pelo código.

## 1. Configuração

In [1]:
import sys, os
os.environ.pop("JAVA_TOOL_OPTIONS", None)
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"   # definido antes da JVM subir, senao o aviso ja saiu
sys.path.insert(0, "../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("tc3")
    .master("local[2]")                      # no AWS Glue esta linha nao existe
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "America/Sao_Paulo")
    .config("spark.ui.showConsoleProgress", "false")   # sem barra de progresso na saida
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/06 02:06:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3


## 2. Por que o código da pergunta não serve como chave

O exemplo abaixo mostra o problema. O mesmo código `2.q` identifica perguntas diferentes conforme a edição.

In [2]:
from schema import ler_cabecalhos

cabecalhos = ler_cabecalhos("../dados/raw")
for edicao, linhas in cabecalhos.items():
    for codigo, descricao, _, _ in linhas:
        if codigo == "2.q":
            print(f"{edicao}  ->  {descricao[:70]}")

2023-2024  ->  Empresa que trabaha passou por layoff em 2023
2024-2025  ->  empresa_passou_por_layoff_em_2024
2025-2026  ->  modelo_de_trabalho_atual


**Leitura do resultado.** Em 2023-2024 e 2024-2025 o código `2.q` trata de demissões em massa. Em 2025-2026 trata do modelo de trabalho. Uma união pelo código colocaria as duas coisas na mesma coluna, sem gerar nenhum erro visível.

Por isso o projeto adota **tripla confirmação**: código, texto da pergunta e conjunto de categorias observado nos dados. Só entra na análise a coluna aprovada nos três critérios.

## 3. Mapa de conceitos de negócio

Cada conceito aponta para a descrição exata da pergunta em cada edição.

In [3]:
from conceitos import CONCEITOS

indice = {ed: {dn: h for c, d, dn, h in linhas} for ed, linhas in cabecalhos.items()}
faltas = [(n, ed) for n, cfg in CONCEITOS.items() for ed, alvo in cfg["desc"].items() if alvo not in indice[ed]]
print(f"conceitos mapeados: {len(CONCEITOS)}")
print(f"conceitos sem correspondencia: {len(faltas)}")

conceitos mapeados: 28
conceitos sem correspondencia: 0


## 4. Regras de harmonização

Cada regra nasceu de uma divergência observada nos dados, nunca de suposição.

In [4]:
import harmonizacao as H

print("booleanos com codificacao divergente:", H.CONCEITOS_BOOLEANOS)
print("correcoes de erro tipografico da pesquisa:")
for conceito, mapa in H.CORRECOES_TIPOGRAFICAS.items():
    for origem, destino in mapa.items():
        print(f"   {conceito}: '{origem}'  ->  '{destino}'")

booleanos com codificacao divergente: ['atua_como_gestor', 'satisfacao']
correcoes de erro tipografico da pesquisa:
   faixa_salarial: 'de R$ 101/mês a R$ 2.000/mês'  ->  'de R$ 1.001/mês a R$ 2.000/mês'
   faixa_salarial: 'de R$ 25.001/mês a R$ 3000/mês'  ->  'de R$ 25.001/mês a R$ 30.000/mês'
   porte_empresa: 'de 501 a 100'  ->  'de 501 a 1.000'


## 5. Execução do Glue Job 01

O mesmo arquivo roda aqui e no AWS Glue.

In [5]:
from job01_bronze_silver import processar

silver, relatorio = processar(spark, "../dados/raw", "../dados/silver")
for r in relatorio:
    print(f"{r['edicao']}: bronze={r['linhas_bronze']}  duplicatas={r['duplicatas_removidas']}  silver={r['linhas_silver']}")
print(f"total: {silver.count()} linhas, {len(silver.columns)} colunas")

2023-2024: bronze=5293  duplicatas=0  silver=5293
2024-2025: bronze=5217  duplicatas=2  silver=5215
2025-2026: bronze=3495  duplicatas=1  silver=3494


total: 14002 linhas, 31 colunas


## 6. Verificação da camada Silver

In [6]:
from pyspark.sql import functions as F

print("booleanos harmonizados:")
silver.groupBy("atua_como_gestor").count().show()

print("erros tipograficos eliminados:")
for coluna, valor in [("faixa_salarial", "de R$ 101/mês a R$ 2.000/mês"), ("porte_empresa", "de 501 a 100")]:
    n = silver.filter(F.col(coluna) == valor).count()
    print(f"   {coluna} = '{valor[:34]}...': {n} ocorrencia(s)")

booleanos harmonizados:


+----------------+-----+
|atua_como_gestor|count|
+----------------+-----+
|             Sim| 2668|
|             Nao|10173|
|            NULL| 1161|
+----------------+-----+

erros tipograficos eliminados:


   faixa_salarial = 'de R$ 101/mês a R$ 2.000/mês...': 0 ocorrencia(s)


   porte_empresa = 'de 501 a 100...': 0 ocorrencia(s)


**Leitura do resultado.** Os booleanos passaram a usar Sim e Não nas três edições, e as duas categorias criadas por erro de digitação na pesquisa de origem foram absorvidas pelas faixas corretas. A camada Silver está pronta para agregação.